# TRIBE v2 Neuromarketing — Colab Demo

Run Meta's TRIBE v2 on your video ads in Google Colab (free T4 GPU).

**Before you start:** `Runtime → Change runtime type → GPU (T4)`.

This notebook will:
1. Clone this project
2. Install deps
3. Log you in to HuggingFace (needed for the gated LLaMA 3.2 backbone)
4. Launch the full Gradio app with a public share link — paste the URL to your marketing friend and they can upload their own videos

License reminder: TRIBE v2 is **CC-BY-NC-4.0**. Non-commercial use only.

In [1]:
# ── 1. Confirm GPU ────────────────────────────────────────────────────────
!nvidia-smi | head -20

Fri Apr 17 02:15:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ── 2. Clone project ──────────────────────────────────────────────────────
# Replace with your own fork if you made one.
import os
REPO_URL = os.environ.get('TRIBE_REPO_URL', 'https://github.com/AnasMK9/tribe-v2-demo.git')
!rm -rf tribe-v2-demo && git clone $REPO_URL tribe-v2-demo
%cd tribe-v2-demo

In [ ]:
# ── 3. Install deps ───────────────────────────────────────────────────────
# Colab has torch pre-installed with CUDA. We only need the rest.
!pip install -q gradio nilearn plotly ffmpeg-python python-dotenv pillow-avif-plugin
!pip install -q git+https://github.com/facebookresearch/tribev2.git
# Nilearn wants spacy models for some operations
!python -m spacy download en_core_web_sm -q || true

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 129.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 407.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 31.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.1/258.1 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.8/122.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 2.0

In [ ]:
# ── 4. HuggingFace login (for LLaMA 3.2-3B access) ────────────────────────
# Create a read token at https://huggingface.co/settings/tokens
# Also: accept the LLaMA 3.2 license on its HF page once before running this.
from huggingface_hub import login
import getpass
token = os.environ.get('HF_TOKEN') or getpass.getpass('HF_TOKEN (hf_…): ')
login(token=token)

In [ ]:
# ── 5. Warm up the model (downloads ~5GB on first run) ────────────────────
import inference
model = inference.get_model()
print('Device:', inference.current_device())

In [ ]:
# ── 6. Option A: Run the full Gradio app (shareable URL) ──────────────────
# This prints a public *.gradio.live URL — share it with your marketing friend.
# The URL lives for 72 hours; re-run this cell to get a new one.
import os
os.environ['GRADIO_SHARE'] = '1'
!python app.py

---
## Option B: Quick inference without the UI

If you just want to check that the pipeline runs end-to-end on a test clip, use the cell below instead of the Gradio launch.

In [ ]:
# Upload a video via the file browser (left sidebar → Files → upload)
# then set this path to it
VIDEO_PATH = 'assets/test_clip.mp4'

import inference, scoring, spikes, viz, interp

preds, t_seconds = inference.predict(VIDEO_PATH)
print('predictions shape:', preds.shape, '· t span:', t_seconds[0], '→', t_seconds[-1], 's')

rts = scoring.aggregate_to_rois(preds, t_seconds)
summary = scoring.summary_scores(rts)
spike_list = spikes.detect_spikes(rts, z_threshold=2.0)

print('\n== SUMMARY ==')
print(interp.overall_verdict(summary['overall'], summary['marketing']))
for k, v in summary['marketing'].items():
    print(f'  {k}: {v:.1f}')

print(f'\n== SPIKES ({len(spike_list)}) ==')
for sp in spike_list[:10]:
    print(' ·', interp.format_spike(sp.cluster, sp.t_peak, sp.peak_z))

In [ ]:
# Show the timeline figure inline
viz.timeline_figure(rts, spike_list)